# 03 · Colombia's Economy by President

| | |
|---|---|
| **Question** | Which Colombian president presided over the strongest economy, and how much does the answer depend on how you measure it? |
| **Data** | World Bank WDI API, IMF World Economic Outlook DataMapper API, minimum wage decrees. Cached copies in `data/` |
| **Skills** | APIs and JSON, Polars, annualizing and per capita comparisons, stress testing a chart from the press |
| **Runs in** | Colab: yes. JupyterLite: yes if Polars loads (uses the cached data) |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericvd-ucb/Colombia-26-NB/blob/main/notebooks/03_colombia_by_president.ipynb) [![Open in JupyterLite](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://ericvd-ucb.github.io/Colombia-26-NB/lab/index.html?path=notebooks/03_colombia_by_president.ipynb)

*Eric Van Dusen, UC Berkeley College of Computing, Data Science, and Society. Prepared for students in Colombia, October 2026.*

In [ ]:
# Setup: run this cell first. It works in Google Colab, JupyterLite, or on your own computer.
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_JUPYTERLITE = sys.platform == "emscripten"

# Saved data lives in the repo's data/ folder; in Colab we download it from GitHub when needed
DATA_URL = "https://raw.githubusercontent.com/ericvd-ucb/Colombia-26-NB/main/data/"
DATA_DIR = "../data" if os.path.isdir("../data") else "data"
os.makedirs(DATA_DIR, exist_ok=True)

def data_path(filename):
    """Local path to a file from the repo's data/ folder, downloading it first if it isn't here."""
    local = os.path.join(DATA_DIR, filename)
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + filename, local)
    return local

if IN_JUPYTERLITE:
    %pip install -q polars

# Colombia's economy by president, 2002 to 2026: who gets credit for what?

Gustavo Petro's term ended on 7 August 2026, so Colombia now has six complete (or nearly complete) four-year terms to compare: Uribe I and II, Santos I and II, Duque, and Petro. Each comparison in the press picks its own indicator, its own years, and its own way of counting:

* A widely shared chart by former finance minister Juan Carlos Echeverry, reported by *Portafolio* (Sept 2024), ranked terms by **cumulative GDP growth**: Santos I on top at 16.3%, Petro's first year at the bottom at 0.4%.
* Duque and Petro argued publicly over who delivered the bigger **real minimum wage** increases. Economists quoted by *El Colombiano* (Jan 2024) said the answer depends on which inflation figure you deflate by.
* End-of-term roundups in August 2026 pointed to **record-low unemployment** and **falling poverty** on one side, and **rising deficits and credit downgrades** on the other.

This notebook sources the data from public APIs, rebuilds these comparisons, and then applies the same four stress tests we used for Spain: **annualize, go per capita, separate nominal from real, and benchmark against peers**. It adds one more that matters a lot for Colombia: **the cycle each president inherited**, especially oil.

The goal is not to grade the presidents. It is to see how framing choices change the story told by the same numbers.

## 1. Sourcing the data

| Indicator | Source | Code |
|---|---|---|
| Real GDP | World Bank | `NY.GDP.MKTP.KD` |
| Real GDP per capita (Colombia and Latin America) | World Bank | `NY.GDP.PCAP.KD` |
| Consumer prices | World Bank | `FP.CPI.TOTL` |
| Unemployment rate (national estimate, DANE) | World Bank | `SL.UEM.TOTL.NE.ZS` |
| Labour force | World Bank | `SL.TLF.TOTL.IN` |
| Population | World Bank | `SP.POP.TOTL` |
| Poverty headcount, national line (DANE) | World Bank | `SI.POV.NAHC` |
| Gini index | World Bank | `SI.POV.GINI` |
| Oil rents, % of GDP | World Bank | `NY.GDP.PETR.RT.ZS` |
| General government gross debt, % of GDP | IMF World Economic Outlook | `GGXWDG_NGDP` |
| Minimum wage (SMMLV) | Government decrees, entered by hand | |

Both APIs are free and need no key. The minimum wage has no clean API, so it is typed in below from the annual decrees, which makes it the one series you should spot-check yourself.

In [ ]:
import json
import urllib.request

import matplotlib.pyplot as plt
import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(200)


def get_json(url):
    """Fetch JSON from an API. Saves a copy to data/; if the live request fails, uses that copy."""
    import hashlib
    filename = "api_" + hashlib.md5(url.encode()).hexdigest()[:12] + ".json"
    try:
        with urllib.request.urlopen(url, timeout=120) as resp:
            js = json.load(resp)
        with open(os.path.join(DATA_DIR, filename), "w") as f:
            json.dump(js, f)
        return js
    except Exception as err:
        print(f"Live request unavailable ({type(err).__name__}); using saved copy {filename}.")
        with open(data_path(filename)) as f:
            return json.load(f)


def worldbank(indicator, countries="COL"):
    # One World Bank indicator for one or more countries (ISO3, ';'-separated) as a tidy frame
    url = (f"https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"
           f"?format=json&per_page=20000&date=1995:2026")
    js = get_json(url)
    rows = [
        {"iso3": d["countryiso3code"] or d["country"]["id"], "year": int(d["date"]), "value": float(d["value"])}
        for d in (js[1] or []) if d["value"] is not None
    ]
    return pl.DataFrame(rows, schema={"iso3": pl.Utf8, "year": pl.Int32, "value": pl.Float64})


def imf(indicator, country="COL"):
    # One IMF World Economic Outlook series from the DataMapper API. Includes IMF projections!
    js = get_json(f"https://www.imf.org/external/datamapper/api/v1/{indicator}/{country}")
    vals = js["values"][indicator][country]
    return pl.DataFrame(
        {"iso3": country, "year": [int(y) for y in vals], "value": [float(v) for v in vals.values()]},
        schema={"iso3": pl.Utf8, "year": pl.Int32, "value": pl.Float64},
    )

In [ ]:
WB = {
    "gdp":    ("NY.GDP.MKTP.KD",    "COL"),
    "gdp_pc": ("NY.GDP.PCAP.KD",    "COL;LCN"),   # LCN = Latin America & Caribbean aggregate
    "cpi":    ("FP.CPI.TOTL",       "COL"),
    "unemp":  ("SL.UEM.TOTL.NE.ZS", "COL"),
    "lf":     ("SL.TLF.TOTL.IN",    "COL"),
    "pop":    ("SP.POP.TOTL",       "COL"),
    "pov":    ("SI.POV.NAHC",       "COL"),
    "gini":   ("SI.POV.GINI",       "COL"),
    "oil":    ("NY.GDP.PETR.RT.ZS", "COL"),
}

raw = {}
for name, (code_, ctry) in WB.items():
    try:
        raw[name] = worldbank(code_, ctry)
        print(f"{name:7s} {code_:20s} {raw[name].height:4d} rows, last year {raw[name]['year'].max()}")
    except Exception as e:
        print(f"{name:7s} {code_:20s} FAILED: {e}")

try:
    raw["debt"] = imf("GGXWDG_NGDP")
    print(f"debt    GGXWDG_NGDP (IMF)    {raw['debt'].height:4d} rows (includes projections)")
except Exception as e:
    print("debt FAILED:", e)

### The minimum wage, by decree

Monthly *salario mínimo mensual legal vigente* in pesos, set each December for the following year. The 2026 value (a 23% increase, Decreto 1469 de 2025) was suspended by the Consejo de Estado in February 2026 and reinstated in July 2026; the nullity case is still open, but the amount did not change.

In [ ]:
smmlv = pl.DataFrame({
    "year": list(range(2000, 2027)),
    "smi": [260_100, 286_000, 309_000, 332_000, 358_000, 381_500, 408_000, 433_700, 461_500,
            496_900, 515_000, 535_600, 566_700, 589_500, 616_000, 644_350, 689_455, 737_717,
            781_242, 828_116, 877_803, 908_526, 1_000_000, 1_160_000, 1_300_000, 1_423_500,
            1_750_905],
}, schema={"year": pl.Int32, "smi": pl.Float64})

smmlv.with_columns((pl.col("smi").pct_change() * 100).round(1).alias("nominal raise %")).tail(10)

### One annual table

In [ ]:
def series(name, col, iso3="COL"):
    return raw[name].filter(pl.col("iso3") == iso3).select("year", pl.col("value").alias(col))


co = pl.DataFrame({"year": pl.int_range(1995, 2027, eager=True).cast(pl.Int32)})
for name, col, iso in [
    ("gdp", "gdp", "COL"), ("gdp_pc", "gdp_pc", "COL"), ("gdp_pc", "gdp_pc_latam", "LCN"),
    ("cpi", "cpi", "COL"), ("unemp", "unemp", "COL"), ("lf", "lf", "COL"), ("pop", "pop", "COL"),
    ("pov", "pov", "COL"), ("gini", "gini", "COL"), ("oil", "oil", "COL"), ("debt", "debt", "COL"),
]:
    if name in raw:
        co = co.join(series(name, col, iso), on="year", how="left")
co = co.join(smmlv, on="year", how="left")

# Last year with actual GDP data. IMF debt beyond this is a projection, so drop it.
LAST = co.filter(pl.col("gdp").is_not_null())["year"].max()
co = co.with_columns(
    pl.when(pl.col("year") <= LAST).then(pl.col("debt")).alias("debt"),
    (pl.col("lf") * (1 - pl.col("unemp") / 100)).alias("emp"),
    (pl.col("smi") / pl.col("cpi") * 100).alias("smi_real"),
    (pl.col("cpi").pct_change() * 100).alias("inflation"),
    (pl.col("gdp").pct_change() * 100).alias("growth"),
)
print("Latest year with GDP data:", LAST)
co.filter(pl.col("year") >= 2000)

### Defining the terms

Colombian presidents take office on 7 August, every four years. With annual data, each transition year is shared by the outgoing and incoming president, the same convention as the Spain notebook. Petro's term runs to 2026, but full-year 2026 data does not exist yet, so his numbers stop at the latest available year. **His term therefore covers fewer data years than the others**, which is exactly why annualizing matters.

In [ ]:
TERMS = [
    ("Uribe I",  2002, 2006),
    ("Uribe II", 2006, 2010),
    ("Santos I", 2010, 2014),
    ("Santos II", 2014, 2018),
    ("Duque",    2018, 2022),
    ("Petro",    2022, 2026),
]


def end_year(col, e):
    # Latest year <= e with data for this column
    avail = co.filter(pl.col(col).is_not_null() & (pl.col("year") <= e))["year"]
    return avail.max() if len(avail) else None


def value(col, year):
    v = co.filter(pl.col("year") == year)[col]
    return v[0] if len(v) and v[0] is not None else None


def change(col, start, end, kind="pct", annualize=False):
    end = end_year(col, end)
    a, b = value(col, start), value(col, end)
    if a is None or b is None or end <= start:
        return None
    years = end - start
    if kind == "pp":
        return (b - a) / years if annualize else b - a
    if annualize:
        return ((b / a) ** (1 / years) - 1) * 100
    return (b / a - 1) * 100


INDICATORS = [
    ("Real GDP", "gdp", "pct"),
    ("Real GDP per capita", "gdp_pc", "pct"),
    ("Prices (CPI)", "cpi", "pct"),
    ("Minimum wage (nominal)", "smi", "pct"),
    ("Minimum wage (real)", "smi_real", "pct"),
    ("Employment", "emp", "pct"),
    ("Unemployment rate (pp)", "unemp", "pp"),
    ("Poverty rate (pp)", "pov", "pp"),
    ("Gini (points)", "gini", "pp"),
    ("Public debt (pp of GDP)", "debt", "pp"),
    ("Population", "pop", "pct"),
]


def term_table(annualize=False):
    data = {"indicator": [lab for lab, _, _ in INDICATORS]}
    for term, s, e in TERMS:
        data[term] = [change(col, s, e, kind, annualize) for _, col, kind in INDICATORS]
    return pl.DataFrame(data).with_columns(pl.exclude("indicator").round(1))


pl.DataFrame({"term": [t for t, _, _ in TERMS],
              "data years": [f"{s} to {end_year('gdp', e)}" for _, s, e in TERMS]})

## 2. The cumulative table (the "newspaper" version)

Percent change from the first to the last year of each term, or change in points for rates. This is the format of the Spanish graphic and most of the Colombian coverage.

In [ ]:
cumulative = term_table()
cumulative

## 3. Rebuilding the Echeverry/Portafolio chart

Here are the figures as *Portafolio* reported them. The article does not say which data or year convention the chart used, and it lists Uribe II as ending in 2010 and Santos I as starting in 2010, so overlapping years are likely. We try two conventions and see which one comes closer:

* **Shared endpoints** (our default): GDP in the term's last year vs its first year.
* **Calendar years in office**: GDP in the last full year vs the year before the first full year (for Uribe I, 2003 to 2006 growth measured from 2002).

In [ ]:
echeverry = {"Uribe I": 14.8, "Uribe II": 9.9, "Santos I": 16.3,
             "Santos II": 3.1, "Duque": 5.9, "Petro": 0.4}  # Petro: first year only


def cum_growth(s, e):
    a, b = value("gdp", s), value("gdp", e)
    return None if a is None or b is None else round((b / a - 1) * 100, 1)


recon = pl.DataFrame({
    "term": [t for t, _, _ in TERMS],
    "Echeverry chart": [echeverry[t] for t, _, _ in TERMS],
    "shared endpoints": [cum_growth(s, end_year("gdp", e)) for _, s, e in TERMS],
    "full calendar years": [cum_growth(s, min(e - 1, LAST)) for _, s, e in TERMS],
    "Petro yr1 check": [None] * 5 + [cum_growth(2022, 2023)],
})
recon

**Question.** Can you reproduce the chart's numbers? If not, what does that tell you about citing a chart from social media in a news story? Notice also that the chart put Petro's *first year* (0.4%) next to other presidents' *full terms*, and that 2023 came right after the 6%+ rebound years of 2021 and 2022.

## 4. Stress test 1: annualize

Four-year terms make Colombia easier than Spain, but Petro's data window is shorter, and a cumulative figure over three years will always look smaller than one over four.

In [ ]:
annual = term_table(annualize=True)
annual

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(co["year"], co["growth"],
       color=["#2c5aa0" if (v or 0) >= 0 else "#c0392b" for v in co["growth"]])
ax.axhline(0, color="black", lw=0.8)
for i, (t, s, e) in enumerate(TERMS):
    if i % 2 == 0:
        ax.axvspan(s, e, color="0.93", zorder=0)
    ax.text((s + e) / 2, ax.get_ylim()[1] * 0.92, t, ha="center", fontsize=9, style="italic")
ax.set_xlim(2000.5, 2026.5)
ax.set_ylabel("Real GDP growth, % y/y")
ax.set_title("Annual GDP growth by term")
plt.show()

## 5. Stress test 2: per capita, and against Latin America

Colombia's population grew steadily, and since 2017 it absorbed close to three million Venezuelan migrants. That raises total output even if output per person does not. We also compare per capita growth with the Latin America and Caribbean aggregate, since commodity prices and global shocks hit the whole region at once.

In [ ]:
bench = pl.DataFrame({
    "term": [t for t, _, _ in TERMS],
    "GDP %/yr": [change("gdp", s, e, annualize=True) for _, s, e in TERMS],
    "GDP pc %/yr": [change("gdp_pc", s, e, annualize=True) for _, s, e in TERMS],
    "LatAm GDP pc %/yr": [change("gdp_pc_latam", s, e, annualize=True) for _, s, e in TERMS],
}).with_columns(
    (pl.col("GDP pc %/yr") - pl.col("LatAm GDP pc %/yr")).alias("Colombia minus LatAm")
).with_columns(pl.exclude("term").round(2))
bench

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(bench["term"], bench["Colombia minus LatAm"],
       color=["#2c5aa0" if (v or 0) >= 0 else "#c0392b" for v in bench["Colombia minus LatAm"]])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("pp per year")
ax.set_title("Colombia's GDP per capita growth relative to Latin America & Caribbean")
plt.show()

## 6. Stress test 3: the minimum wage, nominal vs real

Duque claimed his government delivered the largest real minimum wage increases in decades; Petro disputed it. The disagreement is mostly about **which inflation you subtract**:

* **Last year's inflation.** This is how the raise is actually negotiated each December, and it flatters a year when inflation is *rising* (the raise is set against a lower number than the one that will hit wallets).
* **Same year's inflation.** This measures what happened to purchasing power during the year the wage was in force.

We compute both with annual average CPI. (The official debate uses December-to-December inflation, so expect small differences.)

In [ ]:
mw = co.select(
    "year",
    (pl.col("smi").pct_change() * 100).alias("nominal raise"),
    pl.col("inflation").shift(1).alias("prior-year inflation"),
    pl.col("inflation").alias("same-year inflation"),
).with_columns(
    (((1 + pl.col("nominal raise") / 100) / (1 + pl.col("prior-year inflation") / 100) - 1) * 100)
        .alias("real vs prior-year"),
    (((1 + pl.col("nominal raise") / 100) / (1 + pl.col("same-year inflation") / 100) - 1) * 100)
        .alias("real vs same-year"),
).filter(pl.col("year").is_between(2015, 2026)).with_columns(pl.exclude("year").round(1))
mw

In [ ]:
mw_plot = mw.drop_nulls(["real vs prior-year"])
fig, ax = plt.subplots(figsize=(10, 4))
w = 0.38
ax.bar(mw_plot["year"] - w / 2, mw_plot["real vs prior-year"], w, label="deflated by prior-year inflation", color="#8fb0de")
ax.bar(mw_plot["year"] + w / 2, mw_plot["real vs same-year"].fill_null(0), w, label="deflated by same-year inflation", color="#2c5aa0")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(2022.5, color="0.5", ls="--")
ax.text(2020, ax.get_ylim()[1] * 0.9, "Duque", ha="center")
ax.text(2024.5, ax.get_ylim()[1] * 0.9, "Petro", ha="center")
ax.set_ylabel("real minimum wage increase, %")
ax.set_title("Same raises, two inflation yardsticks")
ax.legend()
plt.show()

**Questions.**
1. In 2022 (decreed by Duque in December 2021, effective under both presidents) inflation spiked. Which method makes that raise look good, and which makes it look bad?
2. Who "owns" a minimum wage that one president decrees in December and the next president governs under from August?
3. The 2026 increase (23%) is by far the largest in the table. What would you need to know to judge whether it raised workers' purchasing power or pushed more workers into informality?

## 7. Stress test 4: the cycle each president inherited

Colombia is an oil exporter. Oil prices collapsed in 2014 to 2015, right as Santos II began, and the commodity boom of the 2000s overlapped Uribe's terms. COVID hit in Duque's second year, and Petro took office during a post-pandemic boom with inflation above 10%. The table below shows conditions **in the year before each term began**, the economy each president received.

In [ ]:
inherited = pl.DataFrame({
    "term": [t for t, _, _ in TERMS],
    "year received": [s - 1 for _, s, _ in TERMS],
    "growth %": [value("growth", s - 1) for _, s, _ in TERMS],
    "inflation %": [value("inflation", s - 1) for _, s, _ in TERMS],
    "unemployment %": [value("unemp", s - 1) for _, s, _ in TERMS],
    "debt % GDP": [value("debt", s - 1) for _, s, _ in TERMS],
    "oil rents % GDP": [value("oil", s - 1) for _, s, _ in TERMS],
}).with_columns(pl.exclude("term", "year received").round(1))
inherited

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.fill_between(co["year"], co["oil"], color="#d9c38c", alpha=0.7, label="Oil rents, % of GDP")
ax1.set_ylabel("Oil rents, % of GDP")
ax2 = ax1.twinx()
ax2.plot(co["year"], co["growth"], color="#2c5aa0", marker="o", ms=3, label="GDP growth")
ax2.axhline(0, color="black", lw=0.6)
ax2.set_ylabel("GDP growth, %")
for t, s, e in TERMS:
    ax1.axvline(s, color="0.6", ls=":", lw=0.8)
    ax1.text(s + 0.1, ax1.get_ylim()[1] * 0.92, t, fontsize=8)
ax1.set_xlim(2000, 2026)
ax1.set_title("The oil cycle and growth")
plt.show()

## 8. The end-of-term scorecard: unemployment, poverty, debt

Petro celebrated July 2026 unemployment of 8.1%, the lowest July since 2001. Critics pointed to two credit downgrades (Fitch in December 2025, S&P in April 2026) and a deficit projected to widen. Both sides are quoting real numbers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col, title, c in [
    (axes[0], "unemp", "Unemployment rate, %", "#c0392b"),
    (axes[1], "pov", "Poverty rate (national line), %", "#7a5195"),
    (axes[2], "debt", "Gross public debt, % of GDP", "#2c5aa0"),
]:
    ax.plot(co["year"], co[col], color=c, marker="o", ms=2.5)
    for _, s, _ in TERMS:
        ax.axvline(s, color="0.8", lw=0.8)
    ax.set_xlim(2000, 2026)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()

latest_u = value("unemp", end_year("unemp", 2026))
prior = co.filter((pl.col("year") < end_year("unemp", 2026)) & (pl.col("unemp") <= latest_u))["year"].max()
print(f"Latest annual unemployment: {latest_u:.1f}%. Last year at or below: {prior}")

**Caveats.**
* The headline "lowest since 2001" refers to a single month compared with the same month in earlier years. Annual averages can tell a slightly different story; check the printout above.
* DANE changed its poverty methodology in 2020 (new survey frame and poverty lines), so pre and post-2020 poverty rates are not strictly comparable. Look for a break in the chart.
* About 55% of Colombian workers are informal, so unemployment alone understates labour market weakness.

## 9. Discussion

1. Rank the six terms on growth three ways: cumulative, annualized, and annualized per capita relative to Latin America. Which president moves the most?
2. Pick one indicator where a president looks best and one where they look worst. Write a one-sentence headline for each. Both should be true.
3. How much of each term's performance do you attribute to oil prices? Design a simple model, such as regressing annual growth on oil rents, and see what is left over by term.
4. Rewrite the Echeverry chart so that it is still one simple bar chart but harder to mislead with.
5. Compare with the Spain notebook: which framing tricks show up in both countries' coverage?